# 13 · Carry or repeat: a learned decision per tile

Every learned model that drew pixels or motion lost to the tile search, or gained a few tenths of
a dB for many times its work: two received frames do not say what happens next, and a network
unsure of it hedges. A **decision** cannot hedge. Per tile it picks one of two sharp pictures - the
search's N+1 (N moved by half the motion it found) or N itself - and a wrong pick is a whole-tile
mistake, never a blur.

The decision sees only what the client has: the search's own work on the two received frames and
the stream's bitrate (`helpers/decide.py`, `FEATURES`).

- **The control, no learning**: the search's gate raised - repeat wherever its cost at no motion is
  under a threshold, tuned on the training clips.
- **Learned**: logistic regression and a small network (12 → 16 → 16 → 1) on the 12 features,
  trained to pick the picture that scores better, each tile weighted by how much better (capped
  at 50 dB, the moving tiles' score) - on every kind of the training split, on the drawn kinds
  only and on the recorded kinds only; the step kept chosen on a held-out tenth of the training
  sources (by video for the recordings).

Scored on the validation split, by kind. Then the decision as the client can run it: the stream's
bitrate and loop filter are not among a graph's two inputs, and the features measured on the picture
cost more in the browser than the search itself - so the same decision on the six features that
come from the search's costs alone, measured inside the graph (`models.Decided`) and exported
beside the search.

Needs: `00_data_preprocess.ipynb`. Exports `extrapolate2x_tilesearch_decide_r4_s2.onnx` into
`dist/frame_gen_extra/`; the tables are cached in `checkpoints/decide/`.

In [1]:
"""Settings."""

import pickle
import time

import numpy as np
import torch

from helpers import decide, models, runs, training
from helpers.decide import FEATURES
from helpers.training import DEVICE

STILL, SHARPNESS, CAP = 0.1, 10.0, 50.0
CACHE = training.CHECKPOINT_DIR / "decide"
SEED = 0
KINDS = ("page", "desktop", "photo", "screen", "video")
DRAWN, RECORDED = ("page", "desktop", "photo"), ("screen", "video")
training.seed_everything(SEED)
manifest = training.load_manifest()
published = models.TileSearch(4, 2, SHARPNESS, 2, 1 / runs.received_every(manifest), still=STILL).to(DEVICE).eval()

## 1. Data

Every client tile at every received frame N from 2 on, of every clip (`runs.valid` frames only):
the 12 features, the wPSNR of repeat and of the search's N+1 over the region the client keeps,
whether the search's gate is open (moving) and whether N+1 differs from N (changed). A tenth of
the training sources - whole videos for the recordings - is held out to choose the training step.

In [2]:
CACHE.mkdir(parents=True, exist_ok=True)


def table(split):
    path = CACHE / f"{split}.pkl"
    if path.exists() and path.stat().st_mtime > (training.DATA_DIR / "manifest.json").stat().st_mtime:
        return pickle.loads(path.read_bytes())
    started = time.time()
    found = decide.load(manifest, split, published)
    path.write_bytes(pickle.dumps(found))
    print(f"{split}: {len(found['repeat'])} tiles in {time.time() - started:.0f}s")
    return found


def subset(found, mask):
    return {key: (value[mask] if isinstance(value, np.ndarray) and len(value) == len(mask) else value)
            for key, value in found.items()}


full = {split: table(split) for split in ("train", "val")}
for split, found in full.items():
    listed = manifest["splits"][split]["sources"]
    found["kind"] = np.array([listed[s]["kind"] for s in found["source"]])
    found["group"] = np.array([listed[s].get("video", listed[s]["file"]) for s in found["source"]])
groups = np.unique(full["train"]["group"])
held = set(np.random.default_rng(SEED).choice(groups, max(1, len(groups) // 10), replace=False))
in_held = np.isin(full["train"]["group"], list(held))
data = {"train": subset(full["train"], ~in_held), "held out": subset(full["train"], in_held), "val": full["val"]}
for name, found in data.items():
    m = found["moving"]
    better = np.minimum(found["search"], CAP) > np.minimum(found["repeat"], CAP)
    worse = np.minimum(found["repeat"], CAP) > np.minimum(found["search"], CAP)
    print(f"{name:<9} {len(m):6} tiles, {m.mean():4.0%} moving; of those the search better on {better[m].mean():4.0%}, "
          f"repeat on {worse[m].mean():4.0%} - " + ", ".join(f"{k} {(found['kind'] == k).sum()}" for k in KINDS))

train      54820 tiles,  61% moving; of those the search better on  57%, repeat on  14% - page 10556, desktop 9860, photo 6496, screen 17353, video 10555
held out    5036 tiles,  61% moving; of those the search better on  57%, repeat on  19% - page 1276, desktop 1276, photo 464, screen 1084, video 936
val        19374 tiles,  56% moving; of those the search better on  49%, repeat on  15% - page 3248, desktop 1392, photo 1740, screen 10652, video 2342


## 2. The threshold

On the moving tiles, capped at 50 dB: the search, repeat, and the search's gate raised - repeat
where the cost at no motion is under a threshold, chosen on the training tiles.

In [3]:
def chosen(found, take_search):
    """(S,) the capped wPSNR of picking the search's N+1 where `take_search`, repeat elsewhere; still tiles repeat."""
    return np.minimum(np.where(take_search & found["moving"], found["search"], found["repeat"]), CAP)


def zero_cost(found):
    return np.exp(found["features"][:, 0]) - 0.01


THRESHOLDS = [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8]
print(f"{'moving tiles, ≤ 50 dB':<26}" + "".join(f"{name:>11}" for name in data))
for label, rule in (("repeat", lambda f: np.zeros(len(f["repeat"]), bool)),
                    ("search (gate 0.1)", lambda f: np.ones(len(f["repeat"]), bool))):
    print(f"{label:<26}" + "".join(f"{chosen(f, rule(f))[f['moving']].mean():11.2f}" for f in data.values()))
print()
for threshold in THRESHOLDS:
    print(f"{f'gate at {threshold} levels':<26}" + "".join(f"{chosen(f, zero_cost(f) >= threshold)[f['moving']].mean():11.2f}"
                                                          for f in data.values()))
GATE = max(THRESHOLDS, key=lambda th: chosen(data["train"], zero_cost(data["train"]) >= th)[data["train"]["moving"]].mean())
print(f"\nchosen on the training tiles: {GATE} levels")

moving tiles, ≤ 50 dB           train   held out        val
repeat                          36.30      34.64      37.62
search (gate 0.1)               37.91      35.34      38.80

gate at 0.1 levels              37.91      35.34      38.80
gate at 0.25 levels             37.91      35.34      38.80
gate at 0.5 levels              37.90      35.34      38.80
gate at 0.75 levels             37.89      35.35      38.80
gate at 1 levels                37.88      35.35      38.78
gate at 1.5 levels              37.84      35.35      38.72
gate at 2 levels                37.81      35.35      38.68
gate at 3 levels                37.71      35.33      38.57
gate at 4 levels                37.64      35.33      38.51
gate at 6 levels                37.53      35.29      38.39
gate at 8 levels                37.41      35.25      38.34

chosen on the training tiles: 0.1 levels


## 3. The learned decision

Inputs: the 12 features (`gap` as log1p), standardised on the training tiles. Target: whether the
search's N+1 scores better than repeat on the tile, capped at 50 dB; weight: by how much. Moving
tiles only - a still tile repeats anyway. Logistic regression and the small network, Adam over the
whole set, kept at the step that scores best on the held-out tiles (moving, capped) - trained on
every kind, on the drawn kinds only and on the recorded kinds only.

In [4]:
def inputs(found, columns=FEATURES):
    """The decision's inputs: the named features, `gap` as log1p."""
    x = found["features"].copy()
    x[:, FEATURES.index("gap")] = np.log1p(x[:, FEATURES.index("gap")])
    return x[:, [FEATURES.index(name) for name in columns]]


def fit(kinds, kind_of_net, columns=FEATURES, steps_max=3000, rate=1e-2):
    """(rule over a table, the module, (held-out score, weights, step), (mean, std) it standardises by)."""
    train = data["train"]
    use = train["moving"] & np.isin(train["kind"], kinds)
    gain = np.minimum(train["search"], CAP) - np.minimum(train["repeat"], CAP)
    x, y, wgt = inputs(train, columns)[use], (gain[use] > 0).astype(np.float32), np.abs(gain[use]).astype(np.float32)
    mean, std = x.mean(0), x.std(0) + 1e-6
    torch.manual_seed(SEED)
    net = torch.nn.Linear(len(columns), 1) if kind_of_net == "logistic" else torch.nn.Sequential(
        torch.nn.Linear(len(columns), 16), torch.nn.ReLU(), torch.nn.Linear(16, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))
    xt, yt, wt = (torch.as_tensor(a, dtype=torch.float32) for a in ((x - mean) / std, y, wgt / wgt.mean()))
    optimiser = torch.optim.Adam(net.parameters(), lr=rate)
    held_out = data["held out"]
    held_x = torch.as_tensor((inputs(held_out, columns) - mean) / std, dtype=torch.float32)
    best = (-np.inf, None, 0)
    for step in range(steps_max + 1):
        if step % 25 == 0:
            with torch.no_grad():
                score = chosen(held_out, (net(held_x)[:, 0] > 0).numpy())[held_out["moving"]].mean()
            if score > best[0] + 1e-4:
                best = (score, {k: v.clone() for k, v in net.state_dict().items()}, step)
        loss = (torch.nn.functional.binary_cross_entropy_with_logits(net(xt)[:, 0], yt, reduction="none") * wt).mean()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    net.load_state_dict(best[1])

    def decide_tiles(found):
        with torch.no_grad():
            return (net(torch.as_tensor((inputs(found, columns) - mean) / std, dtype=torch.float32))[:, 0] > 0).numpy()
    return decide_tiles, net, best, (mean, std)


TRAINED = {}
for label, kinds in (("every kind", KINDS), ("drawn kinds", DRAWN), ("recorded kinds", RECORDED)):
    for kind_of_net in ("logistic", "network"):
        name = f"{kind_of_net}, {label}"
        TRAINED[name] = fit(kinds, kind_of_net)
        print(f"{name:<28} held out: {TRAINED[name][2][0]:.3f} dB (step {TRAINED[name][2][2]})")
held_out = data["held out"]
print(f"{'(the search)':<28} held out: {chosen(held_out, np.ones(len(held_out['repeat']), bool))[held_out['moving']].mean():.3f} dB")
weights = TRAINED["logistic, every kind"][1].weight.detach().numpy()[0]
print("\nlogistic regression on every kind, standardised weights (positive: carry the motion):")
for name, value in sorted(zip(FEATURES, weights), key=lambda p: -abs(p[1])):
    print(f"  {name:<20}{value:+7.2f}")

logistic, every kind         held out: 35.404 dB (step 800)
network, every kind          held out: 35.497 dB (step 125)
logistic, drawn kinds        held out: 35.400 dB (step 1225)
network, drawn kinds         held out: 35.475 dB (step 325)
logistic, recorded kinds     held out: 35.429 dB (step 625)
network, recorded kinds      held out: 35.332 dB (step 225)
(the search)                 held out: 35.343 dB

logistic regression on every kind, standardised weights (positive: carry the motion):
  explained             -0.91
  gap                   +0.76
  log best cost         +0.69
  residual share        -0.32
  change strength       -0.28
  peak                  -0.19
  log zero cost         -0.12
  unexplained cells     -0.07
  log motion            -0.06
  deblock               +0.05
  log bpp               +0.03
  changed pixels        -0.01


## 4. Validation by kind

The validation split - never trained on nor used to choose. Absolute wPSNR on the moving tiles
(capped) per kind and over all; the gain over the search on every moving tile with its 95% range
over videos (clips for the drawn kinds); and the share of moving tiles each rule sends to the search.

In [5]:
rng = np.random.default_rng(SEED)
found = data["val"]
moving, kind, group = found["moving"], found["kind"], found["group"]


def over(gain, mask, draws=2000):
    per = [gain[mask & (group == g)] for g in np.unique(group[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


base = chosen(found, np.ones(len(moving), bool))
ROWS = {}


def header():
    print(f"{'moving tiles, ≤ 50 dB':<54}" + "".join(f"{k:>9}" for k in (*KINDS, "all")) + f"{'over the search':>17}{'95% over videos':>18}{'to search':>11}")
    print(f"{'tiles':<54}" + "".join(f"{(moving & (kind == k)).sum():9}" for k in KINDS) + f"{moving.sum():9}")


def row(name, rule=None):
    """One rule's line of the table - computed once, so a rule shown again keeps its range."""
    if name not in ROWS:
        take = rule(found) & moving
        score = chosen(found, take)
        gain = score - base
        low, high = over(gain, moving) if name != "search (gate 0.1)" else (0, 0)
        ROWS[name] = (f"{name:<54}" + "".join(f"{score[moving & (kind == k)].mean():9.2f}" for k in KINDS) + f"{score[moving].mean():9.2f}"
                      f"{gain[moving].mean():+12.2f}     [{low:+.2f}, {high:+.2f}]{take[moving].mean():10.0%}")
    print(ROWS[name])


RULES = {"repeat": lambda f: np.zeros(len(f["repeat"]), bool),
         "search (gate 0.1)": lambda f: np.ones(len(f["repeat"]), bool),
         f"gate raised to {GATE}": lambda f: zero_cost(f) >= GATE,
         **{f"learned: {name}": TRAINED[name][0] for name in TRAINED}}
header()
for name, rule in RULES.items():
    row(name, rule)

moving tiles, ≤ 50 dB                                      page  desktop    photo   screen    video      all  over the search   95% over videos  to search
tiles                                                      1810      332     1403     5115     2140    10800
repeat                                                    32.06    38.88    31.12    42.74    34.13    37.62       -1.19     [-1.88, -0.49]        0%
search (gate 0.1)                                         34.11    38.29    35.84    43.24    34.20    38.80       +0.00     [+0.00, +0.00]      100%
gate raised to 0.1                                        34.11    38.29    35.84    43.24    34.20    38.80       +0.00     [+0.00, +0.00]      100%
learned: logistic, every kind                             34.41    38.73    36.07    43.26    34.63    38.99       +0.19     [+0.04, +0.43]       58%
learned: network, every kind                              34.57    38.74    36.14    43.25    34.64    39.03       +0.22     [+0.05, +0.

## 5. What the graph can measure

The client's graph gets two frames and nothing else, so the stream's bitrate and loop filter
(`log bpp`, `deblock`) cannot be in it - two of the three features that weigh least. Of the other
ten, four are measured on the picture at full resolution (`residual share`, `unexplained cells`,
`changed pixels`, `change strength`): the luma of both frames, N−2 aligned by a second
`GridSample`, and pooling per cell. A first graph with all ten cost 8.5 ms of GPU a 1080p frame in
the browser, against the search's 2.5 (section 6). The other six come from the search's costs of
its 81 offsets and the motion it found - a few operations on 81 numbers a tile
(`models.Decided.FEATURES`).

The same logistic regression and network, trained on every kind, on the ten and on the six. The
graph takes the six, and of the two models the one that scores better on the held-out tiles.

In [6]:
TEN = tuple(name for name in FEATURES if name not in ("log bpp", "deblock"))
SIX = models.Decided.FEATURES
assert set(SIX) < set(TEN)
for label, columns in (("ten", TEN), ("six, the search's costs", SIX)):
    for kind_of_net in ("logistic", "network"):
        name = f"{kind_of_net}, every kind, {label}"
        TRAINED[name] = fit(KINDS, kind_of_net, columns=columns)
        print(f"{name:<46} held out: {TRAINED[name][2][0]:.3f} dB (step {TRAINED[name][2][2]})")
CHOSEN = max((f"{k}, every kind, six, the search's costs" for k in ("logistic", "network")), key=lambda n: TRAINED[n][2][0])
print(f"to export: {CHOSEN}\n")
header()
for name in ("search (gate 0.1)", "learned: logistic, every kind", "learned: network, every kind"):
    row(name)
for label in ("ten", "six, the search's costs"):
    for kind_of_net in ("logistic", "network"):
        name = f"{kind_of_net}, every kind, {label}"
        row(f"learned: {name}", TRAINED[name][0])

logistic, every kind, ten                      held out: 35.400 dB (step 1425)
network, every kind, ten                       held out: 35.421 dB (step 150)
logistic, every kind, six, the search's costs  held out: 35.395 dB (step 850)
network, every kind, six, the search's costs   held out: 35.423 dB (step 425)
to export: network, every kind, six, the search's costs

moving tiles, ≤ 50 dB                                      page  desktop    photo   screen    video      all  over the search   95% over videos  to search
tiles                                                      1810      332     1403     5115     2140    10800
search (gate 0.1)                                         34.11    38.29    35.84    43.24    34.20    38.80       +0.00     [+0.00, +0.00]      100%
learned: logistic, every kind                             34.41    38.73    36.07    43.26    34.63    38.99       +0.19     [+0.04, +0.43]       58%
learned: network, every kind                              34.57   

## 6. In the graph

`models.Decided` is the published search with the decision inside: the six numbers measured from
the search's own costs in the graph, standardised as in training, the chosen decision, and per
tile the search's N+1 or N - the client's contract exactly, `previous` and `current` in, `output`
out. One part of `decide.features` is rewritten for a graph, to the same number: the next local
minimum of the cost grid, found without a sort.

Checked first on real tiles - the first clips of every kind of the validation split, rebuilt -
against `decide.features` on the same frames (a rebuilt clip is encoded slightly differently from
the cached one, so the table's own rows cannot be compared): the six numbers, the decision they
lead to, and the picture - the decision gates the search's motion, so a repeated tile is the warp
of `current` by no motion, which must be `current` as it is.

In [7]:
import copy

from helpers import clips, webexport
from helpers.training import to_tensor

_, net, _, (mean, std) = TRAINED[CHOSEN]
decided = models.Decided(published, copy.deepcopy(net), mean, std).to(DEVICE).eval()
received = runs.received_every(manifest)
listed = manifest["splits"]["val"]["sources"]
searched = models.Motion(search=published, learned=False).eval()
worst, agree, total, apart, real = np.zeros(len(SIX)), 0, 0, 0.0, []
for kind_name in KINDS:
    for index in [i for i, s in enumerate(listed) if s["kind"] == kind_name][:2]:
        source = listed[index]
        stream = clips.rebuild(source, manifest)
        layout, size = clips.plan(stream.shape[2], stream.shape[1])
        tiles = [(n, t) for n in range(2 * received, len(stream) - 1, received) if runs.valid(source, n, received)
                 for t in range(len(layout))][:128]
        if size != webexport.TILE_INPUT or not tiles:
            continue
        window = np.stack([stream[[n - received, n], layout[t][1]:layout[t][1] + size[0], layout[t][0]:layout[t][0] + size[1]]
                           for n, t in tiles])
        previous, current = to_tensor(window).unbind(1)
        with torch.no_grad():
            table_x = decide.features(published, previous, current, [2.0] * len(tiles), [0.0] * len(tiles)).cpu().numpy()
            graph_x = decided.features(previous, current).cpu().numpy()
            take = decided.logit(previous, current)[:, 0].cpu().numpy() > 0
            # the graph's picture: the search's N+1 where it carries, N itself where it repeats
            expected = torch.where(torch.as_tensor(take, device=DEVICE)[:, None, None, None], searched(previous, current), current)
            apart = max(apart, float((decided(previous, current) - expected).abs().max()))
        worst = np.maximum(worst, np.abs(graph_x - inputs({"features": table_x}, SIX)).max(0))
        agree += (take == TRAINED[CHOSEN][0]({"features": table_x})).sum()
        total += len(tiles)
        real.append((previous[:8].cpu().numpy(), current[:8].cpu().numpy()))
print(f"{total} tiles of {len(real)} clips; the largest difference from decide.features:")
for name, value in zip(SIX, worst):
    print(f"  {name:<20}{value:.1e}")
print(f"the same decision on {agree} of {total} tiles; the picture at most {apart:.1e} from the search's N+1 or N")

880 tiles of 10 clips; the largest difference from decide.features:
  log zero cost       0.0e+00
  log best cost       0.0e+00
  explained           0.0e+00
  peak                0.0e+00
  gap                 4.8e-07
  log motion          1.2e-07
the same decision on 880 of 880 tiles; the picture at most 2.0e-05 from the search's N+1 or N


The graph, into `dist/frame_gen_extra/` twice as notebook 3's search: the client's, with `N`, `H`
and `W` symbolic, and the benchmark page's at a fixed 36 tiles of 328x188 - a 1080p frame. Each
checked against torch on real tiles and on noise, at the shapes the client cuts, and timed on the
desktop CPU beside the published search.

In [8]:
FRAME_TILES = 36
family = "extrapolate2x" if received == 2 else "extrapolate"
name = f"{family}_tilesearch_decide_r4_s2"
label = "tile search ±8 px, stride 2, gated, carry-or-repeat decision"
step_h, step_w = webexport.TILE_STEP
real_previous, real_current = (np.concatenate(part)[:FRAME_TILES] for part in zip(*real))
module = decided.cpu().eval()
for filename, batch, dynamic, tag in ((f"{name}.onnx", 1, True, label),
                                      (f"{name}_tile{step_w}x{step_h}_b{FRAME_TILES}.onnx", FRAME_TILES, False, f"{label} · 1080p frame")):
    info = webexport.export(module, filename, webexport.TILE_INPUT, label=tag, batch=batch, dynamic=dynamic)
    differences = [webexport.check_graph(info["path"], module, real_previous, real_current)]
    for n, h, w in ((FRAME_TILES, *webexport.TILE_INPUT), (4, *webexport.TILE_INPUT), (1, 32, 64))[:3 if dynamic else 1]:
        differences.append(webexport.check_graph(info["path"], module, *np.random.rand(2, n, 3, h, w).astype(np.float32)))
    ms = webexport.time_graph(info["path"], webexport.TILE_INPUT, batch=batch, seconds=2)
    webexport.annotate(info, cpu_ms=round(ms, 2), received_every=received)
    print(webexport.summary_line(info, ms))
    print(f"  as torch on real tiles and noise{' at 36 and 4 tiles of 328x188 and one of 64x32' if dynamic else ''}: "
          f"at most {max(differences):.1e} apart")
searched = webexport.time_graph(webexport.MODELS_DIR / f"{family}_tilesearch_r4_s2.onnx", webexport.TILE_INPUT, seconds=2)
print(f"\nthe published search alone: {searched:.2f} ms a tile on the desktop CPU")
decided.to(DEVICE)

extrapolate2x_tilesearch_decide_r4_s2.onnx            250.3 KB    1.57 ms   Absx1, Addx9, Castx7, Concatx5, Constantx54, ConstantOfShapex1, Convx4, Divx4, Equalx2, Expandx1, Gatherx5, Gemmx3, Greaterx2, GreaterOrEqualx3, GridSamplex1, Lessx1, Logx4, MatMulx2, Mulx10, Rangex2, Reciprocalx2, ReduceMaxx2, ReduceMeanx1, ReduceMinx3, ReduceSumx2, Relux2, Reshapex4, Shapex3, Softmaxx1, Sqrtx1, Subx5, Unsqueezex16, Wherex5   OUTSIDE BUDGET: Abs, Cast, ConstantOfShape, Div, Equal, Expand, Gather, Gemm, Greater, GreaterOrEqual, GridSample, Less, Log, MatMul, Mul, Range, Reciprocal, ReduceMax, ReduceMean, ReduceMin, ReduceSum, Reshape, Shape, Softmax, Sqrt, Sub, Unsqueeze, Where
  as torch on real tiles and noise at 36 and 4 tiles of 328x188 and one of 64x32: at most 1.8e-05 apart
extrapolate2x_tilesearch_decide_r4_s2_tile320x180_b36.onnx  249.8 KB   60.65 ms   Absx1, Addx9, Castx3, Concatx2, Constantx43, ConstantOfShapex1, Convx4, Divx4, Equalx2, Expandx1, Gatherx3, Gemmx3, Greaterx2, GreaterOr

Decided(
  (search): TileSearch(
    (luma): Conv2d(3, 1, kernel_size=(2, 2), stride=(2, 2), bias=False)
    (moved): Conv2d(1, 81, kernel_size=(9, 9), stride=(2, 2), bias=False)
    (centre): Conv2d(1, 1, kernel_size=(9, 9), stride=(2, 2), bias=False)
  )
  (decision): Sequential(
    (0): Linear(in_features=6, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)

**In the browser** (`../upscale/benchmark/main.py --models dist/frame_gen_extra`; Chrome 157
headless on Windows, WebGPU on the RTX 3090, GPU-resident tensors, the output buffer reused; GPU time
from timestamp queries; three runs of 5 s each, the two graphs alternated), one 1080p stream frame -
36 tiles in one batch:

| graph | kernels | GPU ms | wall ms | wall ms, graph capture |
| --- | --- | --- | --- | --- |
| published search (notebook 3) | 32 | 2.51 | 2.58 | 2.41 |
| **search + decision, six features** | **76** | **2.67** | **2.75** | **2.57** |
| first form: ten features (section 5) | 131 | 8.5 | 11.1 | cannot be captured |

The decision is 7% more time. Every node is on the WebGPU provider: the graph runs under graph
capture, and the client's graph keeps on its CPU only the int64 shape arithmetic the published
search keeps there too. The first forms of this graph taught three rules, now in
`helpers/webexport.py`: a `Concat` of more than seven inputs fails (WebGPU allows 8 storage buffers
a kernel); the provider cannot `Flatten`, `Unsqueeze` or `Transpose` a bool tensor and runs those
on its CPU - a round trip from the GPU every run, 8.2 ms of wall clock for 2.7 of GPU; and a
`MaxPool` or a `Where` beside `GridSample` brings in the provider's own layout and a `Transpose`
that ends up on a bool.

With `extrapolate2x_tilesearch_decide_r4_s2.onnx` as its `extrapolate.onnx`, all 16 tests of
desktop-streamer's `tests/enhance.test.js` pass.

## What it says

- **A learned decision is worth +0.2 dB.** On every moving validation tile the small network
  trained on every kind is **+0.22 dB** over the search (95% over videos +0.05 to +0.48) and
  logistic regression +0.19 (+0.04 to +0.43), sending about 60% of the moving tiles to the search.
- **It gains where a motion stops or is not a motion.** By kind, the network is +0.46 dB on pages,
  +0.45 on desktops, +0.44 on video, +0.30 on photos - and +0.01 on screens, where the search and
  repeat are already within 0.5 dB of each other (43.24 against 42.74).
- **Raising the search's gate does nothing.** Repeat wherever its cost at no motion is under a
  threshold, from 0.1 to 8 luma levels: the best is the published 0.1. The tiles where repeat wins
  are not quiet ones; something changed there that one motion gets wrong.
- **What it reads**: how much of the change the best motion explains (`explained`, by far), how
  clear the best match is against the next one (`gap`) and how good it is (`log best cost`), then
  the residual after aligning N−2. It asks the question the search already measures - does one
  motion account for this tile? - and answers "repeat" more readily where the match is ambiguous.
- **It needs every kind to learn from.** Trained on the drawn kinds alone it gains +0.14 (−0.09 to
  +0.41), on the recorded kinds alone the network loses −0.23 and the logistic one gains +0.15:
  each kind alone teaches a rule that does not carry to the other.
- **The search's own costs are enough.** Without the stream's two numbers the network is +0.17
  (+0.02 to +0.38); on the six that come from the search's 81 costs alone, **+0.18 (+0.02 to
  +0.39)** - pages 34.38, desktops 38.72, photos 36.06, screens 43.26, video 34.62 - repeating
  10% of the moving tiles the search would move. What is measured on the picture adds nothing the
  costs do not say, and in the browser it costs more than the search itself.
- **In the graph it is 7% more time**: 2.75 ms of wall clock a 1080p frame against the search's
  2.58, in Chrome's WebGPU on the RTX 3090.

So the decision goes beside the search: `dist/frame_gen_extra/extrapolate2x_tilesearch_decide_r4_s2.onnx`,
the client's contract exactly, +0.18 dB over the published search on the moving tiles for six
numbers its costs give and a 400-weight network, no picture drawn that is not the search's or N,
no state kept between frames. It cannot see a stop that is not announced: what happens between N
and N+1 is not in the two frames.